## Config paths/models

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import json
import os
import sys


#PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/")
#DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")

PROJECT_ROOT = Path("/home/victor/gw/Gravitational-Waves-Lab/cbc_pe/")
DATA_ROOT = PROJECT_ROOT / "data"


DATA_PROCESSED = DATA_ROOT / "processed"
RESULTS_DIR = DATA_ROOT / "results"
MODELS_DIR = DATA_ROOT / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"

checkpoint_id = (
    "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
    "_SimpleCNN_Pool"
    "_emb128_pool1"
    "_MSELoss"
    "_seed123"
)

pred_path = RESULTS_DIR / dataset_id / f"{checkpoint_id}_train_val_predictions_embeddings.npz"
history_path = RESULTS_DIR / dataset_id / f"{checkpoint_id}_history.npz"
checkpoint_path = CHECKPOINTS_DIR / dataset_id / f"{checkpoint_id}_checkpoint.pt"

assert pred_path.exists(), pred_path

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT does not exist: {PROJECT_ROOT}")

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"'src' directory not found inside PROJECT_ROOT: {PROJECT_ROOT / 'src'}")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", Path.cwd())
print("pred_path:", pred_path)
print("project_root exist:", PROJECT_ROOT.exists())
print("history_path exists:", history_path.exists())
print("checkpoint_path exists:", checkpoint_path.exists())

cwd: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe
pred_path: /home/victor/gw/Gravitational-Waves-Lab/cbc_pe/data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n100_000/bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Pool_emb128_pool1_MSELoss_seed123_train_val_predictions_embeddings.npz
project_root exist: True
history_path exists: True
checkpoint_path exists: True


## Load the preds(/embs)

In [2]:
from pathlib import Path
import numpy as np

#LOCAL_DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")

LOCAL_DATA_ROOT = DATA_ROOT

DATA_PROCESSED = LOCAL_DATA_ROOT / "processed"

data = np.load(pred_path, allow_pickle=True)

pred_train = data["pred_train"]
y_train = data["y_train"]
emb_train = data["emb_train"]

pred_val = data["pred_val"]
y_val = data["y_val"]
emb_val = data["emb_val"]

y_mean = data["y_mean"]
y_std = data["y_std"]

train_idx = data["train_idx"]
val_idx = data["val_idx"]

label_names = data["label_names"].tolist()

# Ruta guardada durante entrenamiento, posiblemente de otra máquina
stored_dataset_path = Path(str(data["dataset_path"]))
stored_split_path = Path(str(data["split_path"]))
stored_label_stats_path = Path(str(data["label_stats_path"]))

print("stored_dataset_path:", stored_dataset_path)

# Reconstruir ruta local
dataset_path = DATA_PROCESSED / dataset_id / stored_dataset_path.name
split_path = DATA_PROCESSED / dataset_id / stored_split_path.name
label_stats_path = DATA_PROCESSED / dataset_id / stored_label_stats_path.name

print("local dataset_path:", dataset_path)
print("local split_path:", split_path)
print("local label_stats_path:", label_stats_path)

assert dataset_path.exists(), dataset_path

KeyError: 'train_idx is not a file in the archive'

## (Standarized) Metrics

In [ ]:
from src.models.evaluate import regression_metrics, inverse_standardize

metrics_train_std = regression_metrics(
    y_true=y_train,
    y_pred=pred_train,
    label_names=label_names,
    split_name="train_std",
)

metrics_val_std = regression_metrics(
    y_true=y_val,
    y_pred=pred_val,
    label_names=label_names,
    split_name="val_std",
)

metrics_all_std = pd.concat(
    [metrics_train_std, metrics_val_std],
    ignore_index=True,
)

metrics_all_std

In [ ]:
print("train global MSE:", metrics_train_std["MSE"].mean())
print("val global MSE:", metrics_val_std["MSE"].mean())

## (Physical) Metrics

In [ ]:
pred_train_phys = inverse_standardize(pred_train, y_mean, y_std)
y_train_phys = inverse_standardize(y_train, y_mean, y_std)

pred_val_phys = inverse_standardize(pred_val, y_mean, y_std)
y_val_phys = inverse_standardize(y_val, y_mean, y_std)

metrics_train_phys = regression_metrics(
    y_true=y_train_phys,
    y_pred=pred_train_phys,
    label_names=label_names,
    split_name="train_phys",
)

metrics_val_phys = regression_metrics(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys",
)

metrics_all_phys = pd.concat(
    [metrics_train_phys, metrics_val_phys],
    ignore_index=True,
)

metrics_all_phys

## Training curve

In [ ]:
if history_path.exists():
    history = np.load(history_path, allow_pickle=True)

    print(history.files)

    train_loss = history["train_loss"]
    val_loss = history["val_loss"]

    plt.figure(figsize=(7, 4))
    plt.plot(train_loss, label="train")
    plt.plot(val_loss, label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print("best val loss from history:", np.min(val_loss))
else:
    print("No history file found:", history_path)

## Evaluation Plots

In [ ]:
from src.models.plots import plot_pred_vs_true_density, plot_residuals, plot_mean_abs_error_vs_true_pred, plot_residual_vs_true_density, plot_abs_error_vs_quantity_density, plot_binned_error_vs_quantity

### Pred vs True

In [ ]:
plot_pred_vs_true_density(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys",
)


### Residuals

In [ ]:
plot_residuals(y_val_phys, pred_val_phys, label_names, "val_phys")
plot_residual_vs_true_density(y_val_phys, pred_val_phys, label_names, "val_phys")

In [ ]:
plot_mean_abs_error_vs_true_pred(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys",
)

### Error vs SNR

In [ ]:
with h5py.File(dataset_path, "r") as f:
    print(list(f.keys()))
    print(list(f["snr"].keys()))

    snr_all = f["snr/network"][:]
    snr_train = snr_all[train_idx]
    snr_val = snr_all[val_idx]

print("snr_train:", snr_train.min(), snr_train.max())
print("snr_val:", snr_val.min(), snr_val.max())

In [ ]:
plot_abs_error_vs_quantity_density(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="network SNR",
    split_name="val_phys",
)

### Error vs parameters (mass, distance, ...)

In [ ]:
with h5py.File(dataset_path, "r") as f:
    mass_1_all = f["parameters/mass_1"][:]
    mass_2_all = f["parameters/mass_2"][:]
    distance_all = f["parameters/distance"][:]

mass_1_val = mass_1_all[val_idx]
mass_2_val = mass_2_all[val_idx]
distance_val = distance_all[val_idx]

In [ ]:
from src.models.plots import plot_abs_error_vs_quantity_density

plot_abs_error_vs_quantity_density(
    quantity=distance_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="distance",
    split_name="val_phys",
)

For all labels:

In [ ]:
for j, label in enumerate(label_names):
    plot_abs_error_vs_quantity_density(
        quantity=y_val_phys[:, j],
        y_true=y_val_phys,
        y_pred=pred_val_phys,
        label_names=label_names,
        quantity_name=f"true {label}",
        split_name="val_phys",
    )

### Binned error

In [ ]:
plot_binned_error_vs_quantity(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="network SNR",
    split_name="val_phys",
    n_bins=10,
)

USEFUL PLOTS 
_________________________

1. pred vs true density
2. residual vs true density
3. binned abs error vs SNR
4. binned abs error vs true chirp_mass / total_mass / chi_eff

## Save the metrics (cvs)

In [ ]:
eval_dir = RESULTS_DIR / dataset_id / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics_std_path = eval_dir / f"{checkpoint_id}_metrics_std.csv"
metrics_phys_path = eval_dir / f"{checkpoint_id}_metrics_phys.csv"

metrics_all_std.to_csv(metrics_std_path, index=False)
metrics_all_phys.to_csv(metrics_phys_path, index=False)

print("Saved:", metrics_std_path)
print("Saved:", metrics_phys_path)

## Comparing models

In [ ]:
from src.models.utils import summarize_prediction_file

prediction_files = sorted((RESULTS_DIR / dataset_id ).glob("*train_val_predictions_embeddings.npz"))

print("Found prediction files:", len(prediction_files))
for p in prediction_files:
    print("  ", p.name)

if len(prediction_files) == 0:
    raise FileNotFoundError(
        f"No prediction files found in {RESULTS_DIR} "
        "with pattern '*train_val_predictions_embeddings.npz'."
    )

summary_rows = [summarize_prediction_file(p) for p in prediction_files]

summary_df = pd.DataFrame(summary_rows)

print("Summary columns:", summary_df.columns.tolist())

if "val_MSE_global" not in summary_df.columns:
    raise KeyError(
        "'val_MSE_global' not found in summary_df. "
        "Check summarize_prediction_file(). "
        f"Available columns: {summary_df.columns.tolist()}"
    )

summary_df = summary_df.sort_values("val_MSE_global")

summary_df
